# Camada Gold — Indicadores e Análises

## Objetivo

Transformar os dados tratados na Silver em métricas e tabelas analíticas para investigar o perfil dos passageiros e os fatores associados à sobrevivência no Titanic.

## Análises propostas

1. **Indicadores gerais:** total de passageiros, sobreviventes, não sobreviventes, percentuais e idade média geral.
2. **Análise por sexo:** quantidade de passageiros, idade média e taxa de sobrevivência.
3. **Sobrevivência por faixa etária:** comparar as taxas de sobrevivência entre grupos de idade.
4. **Sobrevivência por classe:** comparar as taxas entre as classes de passageiros.
5. **Sobrevivência por porto de embarque:** analisar os resultados por local de embarque.
6. **Sobrevivência por deck:** comparar os resultados conforme o deck informado, considerando a categoria desconhecida.
7. **Sobrevivência por classe e sexo:** investigar diferenças entre grupos combinados.

## Origem dos dados

Tabela Delta: `projeto_titanic.silver.titanic`

## Resultado esperado

Criar tabelas Delta na camada Gold, com métricas consistentes e adequadas à análise exploratória e à construção de um dashboard no Power BI.


In [0]:
#Importação dos dados da tabela silver para um dataframe

from pyspark.sql.functions import *

df_silver_titanic = spark.table("projeto_titanic.silver.titanic")

display(df_silver_titanic.limit(5))

classe,sexo,id_passageiro,sobreviveu,nome,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto_embarque,deck
3,masculino,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,Southampton,U
1,feminino,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",38.0,1,0,PC 17599,71.2833,Cherbourg,C
3,feminino,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,Southampton,U
1,feminino,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,Southampton,C
3,masculino,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,Southampton,U


In [0]:
# 1. Indicadores gerais dos passageiros

df_gold_indicadores_gerais = (
    df_silver_titanic
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes"),
        avg("idade").alias("idade_media_geral")
    )
    .withColumn(
        "percentual_sobreviventes",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .withColumn(
        "percentual_nao_sobreviventes",
        round(col("total_nao_sobreviventes") / col("total_passageiros") * 100, 2)
    )
)

display(df_gold_indicadores_gerais)

total_passageiros,total_sobreviventes,total_nao_sobreviventes,idade_media_geral,percentual_sobreviventes,percentual_nao_sobreviventes
891,342,549,29.088855218855215,38.38,61.62


In [0]:
# 2. Indicadores por sexo

df_gold_analise_por_sexo = (
    df_silver_titanic
    .groupBy("sexo")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes"),
        avg("idade").alias("idade_media")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
)

display(df_gold_analise_por_sexo)

sexo,total_passageiros,total_sobreviventes,total_nao_sobreviventes,idade_media,taxa_sobrevivencia
masculino,577,109,468,30.119878682842284,18.89
feminino,314,233,81,27.194267515923567,74.2


In [0]:
# 3. Sobrevivência por faixa etária

df_gold_sobrevivencia_faixa_etaria = (
    df_silver_titanic
    .withColumn(
        "faixa_etaria",
        when(col("idade") <= 12, "00-12 | Crianças")
        .when(col("idade") <= 17, "13-17 | Adolescentes")
        .when(col("idade") <= 29, "18-29 | Adultos jovens")
        .when(col("idade") <= 59, "30-59 | Adultos")
        .otherwise("60+ | Idosos")
    )
    .groupBy("faixa_etaria")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("faixa_etaria")
)

display(df_gold_sobrevivencia_faixa_etaria)

faixa_etaria,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
00-12 | Crianças,69,40,29,57.97
13-17 | Adolescentes,44,21,23,47.73
18-29 | Adultos jovens,409,131,278,32.03
30-59 | Adultos,343,143,200,41.69
60+ | Idosos,26,7,19,26.92


In [0]:
# 4. Sobrevivência por classe

df_gold_sobrevivencia_classe = (
    df_silver_titanic
    .groupBy("classe")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("classe")
)

display(df_gold_sobrevivencia_classe)

classe,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
1,216,136,80,62.96
2,184,87,97,47.28
3,491,119,372,24.24


In [0]:
# 5. Sobrevivência por porto de embarque

df_gold_sobrevivencia_porto = (
    df_silver_titanic
    .groupBy("porto_embarque")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("porto_embarque")
)

display(df_gold_sobrevivencia_porto)

porto_embarque,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
Cherbourg,168,93,75,55.36
Queenstown,77,30,47,38.96
Southampton,646,219,427,33.9


In [0]:
# 6. Sobrevivência por deck
# U representa cabines sem informação de deck (desconhecido).

df_gold_sobrevivencia_deck = (
    df_silver_titanic
    .groupBy("deck")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("deck")
)

display(df_gold_sobrevivencia_deck)

deck,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
A,15,7,8,46.67
B,47,35,12,74.47
C,59,35,24,59.32
D,33,25,8,75.76
E,32,24,8,75.0
F,13,8,5,61.54
G,4,2,2,50.0
T,1,0,1,0.0
U,687,206,481,29.99


In [0]:
# 7. Sobrevivência por classe e sexo

df_gold_sobrevivencia_classe_sexo = (
    df_silver_titanic
    .groupBy("classe", "sexo")
    .agg(
        count("*").alias("total_passageiros"),
        sum(when(col("sobreviveu") == 1, 1).otherwise(0)).alias("total_sobreviventes"),
        sum(when(col("sobreviveu") == 0, 1).otherwise(0)).alias("total_nao_sobreviventes")
    )
    .withColumn(
        "taxa_sobrevivencia",
        round(col("total_sobreviventes") / col("total_passageiros") * 100, 2)
    )
    .orderBy("classe", "sexo")
)

display(df_gold_sobrevivencia_classe_sexo)

classe,sexo,total_passageiros,total_sobreviventes,total_nao_sobreviventes,taxa_sobrevivencia
1,feminino,94,91,3,96.81
1,masculino,122,45,77,36.89
2,feminino,76,70,6,92.11
2,masculino,108,17,91,15.74
3,feminino,144,72,72,50.0
3,masculino,347,47,300,13.54


In [0]:
# Gravação das tabelas analíticas na camada Gold

(
    df_gold_indicadores_gerais.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.indicadores_gerais")
)

(
    df_gold_analise_por_sexo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.analise_por_sexo")
)

(
    df_gold_sobrevivencia_faixa_etaria.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.sobrevivencia_por_faixa_etaria")
)

(
    df_gold_sobrevivencia_classe.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.sobrevivencia_por_classe")
)

(
    df_gold_sobrevivencia_porto.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.sobrevivencia_por_porto_embarque")
)

(
    df_gold_sobrevivencia_deck.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.sobrevivencia_por_deck")
)

(
    df_gold_sobrevivencia_classe_sexo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("projeto_titanic.gold.sobrevivencia_por_classe_sexo")
)

In [0]:
# Conferência das tabelas criadas na camada Gold

display(
    spark.sql("SHOW TABLES IN projeto_titanic.gold")
)

database,tableName,isTemporary
gold,analise_por_sexo,false
gold,indicadores_gerais,false
gold,sobrevivencia_por_classe,false
gold,sobrevivencia_por_classe_sexo,false
gold,sobrevivencia_por_deck,false
gold,sobrevivencia_por_faixa_etaria,false
gold,sobrevivencia_por_porto_embarque,false
